## YOLOv8 Model for Smoke Detection in Coal Combustion

In [1]:
# Mount Google Drive to access the dataset
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
# Install ultralytics library for YOLOv8
!pip3 install ultralytics

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.6/46.6 kB 1.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 12.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.2/53.2 kB 1.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 77.4/77.4 kB 1.7 MB/s eta 0:00:00


In [7]:
# Define the path to your dataset in Google Drive
dataset_path = '/content/drive/MyDrive/coal_combustion_image_dataset'

# Create a YAML file for YOLOv8 data configuration
data_yaml_content = f"""
path: {dataset_path}  # dataset root dir
train: train/images  # train images (relative to 'path')
val: valid/images   # val images (relative to 'path')
test: test/images   # test images (optional)

# Classes
nc: 1  # number of classes
names: ['smoke']  # class names
"""

# Save the YAML content to a file
with open('data.yaml', 'w') as f:
    f.write(data_yaml_content)

print("data.yaml created successfully:")
print(data_yaml_content)

data.yaml created successfully:

path: /content/drive/MyDrive/coal_combustion_image_dataset  # dataset root dir
train: train/images  # train images (relative to 'path')
val: valid/images   # val images (relative to 'path')
test: test/images   # test images (optional)

# Classes
nc: 1  # number of classes
names: ['smoke']  # class names



In [8]:
from ultralytics import YOLO

# Loaded a pre-trained YOLOv8n model (nano version)
model = YOLO('yolov8n.pt')

# Train the model
# epochs: Number of epochs to train for
# img_size: Size of input images as integer or w,h
# data: Path to data.yaml file
results = model.train(data='data.yaml', epochs=25, imgsz=640)

Ultralytics 8.4.153 🚀 Python-3.13.15 torch-2.11.0+cpu CPU (Intel Xeon CPU @ 2.20GHz)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=data.yaml, degrees=0.0, deterministic=True, device=, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=25, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=train-2, nbs=64, nms=None, opset=None, optimize=False, optimizer=a

In [12]:
# Validate the model on the test set
metrics = model.val(data='data.yaml')  # This will use the 'test' split defined in data.yaml if available, otherwise 'val'

# Print detailed metrics
print("\n--- Model Validation Results (Test Set) ---")
print(f"Precision (P): {metrics.results_dict['metrics/precision(B)']: .4f}")
print(f"Recall (R): {metrics.results_dict['metrics/recall(B)']: .4f}")
print(f"mAP50: {metrics.results_dict['metrics/mAP50(B)']: .4f}")
print(f"mAP50-95: {metrics.results_dict['metrics/mAP50-95(B)']: .4f}")
print(f"F1 Score: {2 * (metrics.results_dict['metrics/precision(B)'] * metrics.results_dict['metrics/recall(B)']) / (metrics.results_dict['metrics/precision(B)'] + metrics.results_dict['metrics/recall(B)']): .4f}")

Ultralytics 8.4.153 🚀 Python-3.13.15 torch-2.11.0+cpu CPU (Intel Xeon CPU @ 2.20GHz)
Model summary (fused): 72 layers, 3,005,843 parameters, 0 gradients, 8.1 GFLOPs
WARNING ⚠️ val: Slow image access detected (ping: 0.5±0.1 ms, read: 34.1±13.9 MB/s, size: 41.1 KB). Use local storage instead of remote/mounted storage for better performance. See https://docs.ultralytics.com/guides/model-training-tips
val: Scanning /content/drive/MyDrive/coal_combustion_image_dataset/valid/labels.cache... 52 images, 19 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 52/52 10.4Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 4/4 3.1s/it 12.5s
                   all         52         43      0.807      0.512      0.688      0.386
Speed: 2.5ms preprocess, 210.9ms inference, 0.0ms loss, 2.0ms postprocess per image
Results saved to /content/runs/detect/val-3

--- Model Validation Results (Test Set) ---
Precision (P):  0.8071
Recall (R):  0.5116
mA

In [13]:
# Export the trained model
# Export to ONNX format (a common format for deployment)
export_path = model.export(format='onnx')
print(f"Model exported to: {export_path}")

# You can also save the entire model object if you wish to load it later within Python
import torch
torch.save(model.state_dict(), 'yolov8n_smoke_detector.pt')
print("Model state dictionary saved as yolov8n_smoke_detector.pt")


Ultralytics 8.4.153 🚀 Python-3.13.15 torch-2.11.0+cpu CPU (Intel Xeon CPU @ 2.20GHz)
Model summary (fused): 72 layers, 3,005,843 parameters, 0 gradients, 8.1 GFLOPs

PyTorch: starting from '/content/runs/detect/train-2/weights/best.pt' with input shape (1, 3, 640, 640) BCHW and output shape(s) (1, 5, 8400) (6.0 MB)

ONNX: starting export with onnx 1.22.0 opset 18...
ONNX: slimming with onnxslim 0.1.96...
ONNX: export success ✅ 1.6s, saved as '/content/runs/detect/train-2/weights/best.onnx' (11.7 MB)

Export complete (2.4s)
Results saved to /content/runs/detect/train-2/weights/best.onnx
Predict:         yolo predict task=detect model=/content/runs/detect/train-2/weights/best.onnx imgsz=640 
Validate:        yolo val task=detect model=/content/runs/detect/train-2/weights/best.onnx imgsz=640 data=data.yaml  
Visualize:       https://netron.app
Model exported to: /content/runs/detect/train-2/weights/best.onnx
Model state dictionary saved as yolov8n_smoke_detector.pt
